# Análisis exploratorio de House Prices

Este cuaderno analiza el precio de venta de viviendas en Ames, Iowa. Complementa el script de preparación y concentra tablas, gráficas e interpretaciones para comunicar los hallazgos.

## Objetivos

- Describir la estructura, tipos de datos y faltantes.
- Separar `SalePrice` como variable dependiente.
- Analizar tendencia central, dispersión, posición, sesgo y curtosis.
- Explorar correlaciones, frecuencias, histogramas y diagramas de caja.
- Preparar variables categóricas mediante códigos y variables dummy.

## Dataset seleccionado

[Kaggle](https://www.kaggle.com/) es una plataforma de datos que permite publicar, documentar y explorar conjuntos de datos. El análisis utiliza el [House Prices dataset](https://www.kaggle.com/datasets/lespin/house-prices-dataset), con ventas históricas de viviendas en Ames, Iowa.

El archivo `train.csv` contiene 1,460 viviendas y 81 variables. `SalePrice` es la variable numérica dependiente y representa el precio de venta en dólares. Tras excluir `Id`, se analizan 36 predictores numéricos y 43 categóricos; `LotFrontage`, `MasVnrArea` y `GarageYrBlt` tienen faltantes numéricos.

Entre las variables relevantes se encuentran `OverallQual` (calidad general), `GrLivArea` (superficie habitable), `YearBuilt` (año de construcción), `Neighborhood` (zona), `HouseStyle` (estilo de vivienda) y `SaleCondition` (condición de venta). El diccionario completo de las variables está disponible en `data/raw/house-prices/data_description.txt`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Permite ejecutar el cuaderno desde la raíz o desde notebooks/.
ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'raw' / 'house-prices' / 'train.csv'
TARGET = 'SalePrice'
IDENTIFIER = 'Id'

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 100)

In [ ]:
# Lee celdas vacías y la etiqueta NA como valores faltantes.
df = pd.read_csv(DATA_PATH, na_values=['NA'])

print(f'Registros: {df.shape[0]:,}')
print(f'Variables: {df.shape[1]}')
display(df.head())
display(df.dtypes.value_counts().rename_axis('tipo').to_frame('cantidad'))

## Perfil y calidad de los datos

In [ ]:
# Separa el identificador, los predictores y la variable objetivo.
y = df[TARGET].copy()
X = df.drop(columns=[TARGET, IDENTIFIER]).copy()
numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()

profile = pd.DataFrame({
    'registros': [len(df)],
    'predictores': [X.shape[1]],
    'numéricos': [len(numeric_cols)],
    'categóricos': [len(categorical_cols)],
    'duplicados': [int(df.duplicated().sum())],
})
display(profile)

# Cuenta los registros disponibles y ausentes en cada variable.
value_counts = pd.DataFrame({
    'valores_no_nulos': df.count(),
    'faltantes': df.isna().sum(),
})
display(value_counts)

missing = (df.isna().sum()
           .loc[lambda s: s.gt(0)]
           .sort_values(ascending=False)
           .rename('faltantes')
           .to_frame())
missing['porcentaje'] = missing['faltantes'] / len(df) * 100
display(missing)

In [ ]:
# Visualiza las variables con mayor proporción de datos ausentes.
plot_missing = missing.head(20).sort_values('porcentaje')
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(plot_missing.index, plot_missing['porcentaje'], color='#2e75b6')
ax.set(title='Variables con valores faltantes', xlabel='Porcentaje de registros')
plt.show()

## Preparación de predictores

La imputación se aplica únicamente a predictores numéricos. Las categorías ausentes se identifican como `Sin_dato` antes de crear códigos o variables dummy.

In [ ]:
# Calcula y aplica la media por cada predictor numérico con faltantes.
numeric_means = X[numeric_cols].mean()
X_imputed = X.copy()
X_imputed[numeric_cols] = X_imputed[numeric_cols].fillna(numeric_means)

imputation_check = pd.DataFrame({
    'faltantes_antes': X[numeric_cols].isna().sum(),
    'media_utilizada': numeric_means,
    'faltantes_después': X_imputed[numeric_cols].isna().sum(),
}).query('faltantes_antes > 0')
display(imputation_check)

# Genera las dos representaciones solicitadas de variables categóricas.
X_codes = X_imputed.copy()
for column in categorical_cols:
    X_codes[column] = pd.factorize(X_codes[column].fillna('Sin_dato').astype(str), sort=True)[0]

X_dummies = pd.get_dummies(
    X_imputed.assign(**{c: X_imputed[c].fillna('Sin_dato') for c in categorical_cols}),
    columns=categorical_cols,
    dtype=int,
)

print(f'Dimensión de X: {X.shape}')
print(f'Dimensión de y: {y.shape}')
print(f'Columnas con códigos: {X_codes.shape[1]}')
print(f'Columnas dummy: {X_dummies.shape[1]}')

# Muestra evidencia de una codificación y de las variables dummy resultantes.
coding_example = pd.DataFrame({
    'Neighborhood_original': X['Neighborhood'].fillna('Sin_dato'),
    'Neighborhood_codigo': X_codes['Neighborhood'],
}).drop_duplicates().sort_values('Neighborhood_codigo')
display(coding_example)
display(X_dummies.filter(regex=r'^Neighborhood_').head())

## Estadística descriptiva de variables cuantitativas

In [ ]:
# Resume tendencia central, dispersión, posición y forma de la variable objetivo.
saleprice_summary = pd.DataFrame({
    'medida': ['n', 'media', 'mediana', 'moda', 'mínimo', 'Q1', 'Q3', 'máximo',
               'rango', 'varianza', 'desviación estándar', 'CV (%)', 'sesgo', 'curtosis'],
    'valor': [
        y.count(), y.mean(), y.median(), y.mode().iat[0], y.min(),
        y.quantile(.25), y.quantile(.75), y.max(), y.max() - y.min(),
        y.var(), y.std(), y.std() / y.mean() * 100, y.skew(), y.kurt(),
    ],
})
display(saleprice_summary.style.format({'valor': '{:,.2f}'}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(y, bins=35, kde=True, ax=axes[0], color='#2e75b6')
axes[0].set(title='Distribución de SalePrice', xlabel='Precio de venta (USD)')
sns.boxplot(x=y, ax=axes[1], color='#8cbce4')
axes[1].set(title='Diagrama de caja de SalePrice', xlabel='Precio de venta (USD)')
plt.tight_layout()
plt.show()

In [ ]:
# Resume todas las variables cuantitativas con las medidas solicitadas.
quantitative_data = pd.concat([X_imputed[numeric_cols], y], axis=1)
numeric_summary = pd.DataFrame({
    'valores_no_nulos': quantitative_data.count(),
    'media': quantitative_data.mean(),
    'mediana': quantitative_data.median(),
    'mínimo': quantitative_data.min(),
    'Q1': quantitative_data.quantile(.25),
    'Q3': quantitative_data.quantile(.75),
    'máximo': quantitative_data.max(),
    'rango': quantitative_data.max() - quantitative_data.min(),
    'varianza': quantitative_data.var(),
    'desviación_estándar': quantitative_data.std(),
    'sesgo': quantitative_data.skew(),
    'curtosis': quantitative_data.kurt(),
})
numeric_summary['moda'] = quantitative_data.mode().iloc[0]
numeric_summary['CV (%)'] = (numeric_summary['desviación_estándar'] / numeric_summary['media']) * 100
numeric_summary = numeric_summary[[
    'valores_no_nulos', 'media', 'mediana', 'moda', 'mínimo', 'Q1', 'Q3', 'máximo',
    'rango', 'varianza', 'desviación_estándar', 'CV (%)', 'sesgo', 'curtosis',
]]
display(numeric_summary.style.format('{:,.2f}'))

In [ ]:
# Compara la forma de SalePrice con dos medidas físicas relevantes.
histogram_columns = [TARGET, 'GrLivArea', 'TotalBsmtSF']
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, column in zip(axes, histogram_columns):
    sns.histplot(df[column], bins=30, kde=True, ax=ax, color='#2e75b6')
    ax.set(title=f'Distribución de {column}', xlabel=column)
plt.tight_layout()
plt.show()

## Correlaciones y comparaciones por categoría

In [ ]:
# Calcula asociaciones lineales entre SalePrice y los predictores numéricos.
numeric_analysis = quantitative_data.copy()
correlations = (numeric_analysis.corr(numeric_only=True)[TARGET]
                .drop(TARGET)
                .sort_values(key=lambda s: s.abs(), ascending=False)
                .rename('correlación con SalePrice')
                .to_frame())
display(correlations.head(15))

# Lista los pares cuantitativos con mayor relación lineal absoluta.
correlation_matrix = numeric_analysis.corr()
pairwise_correlations = correlation_matrix.stack().rename('correlación').reset_index()
pairwise_correlations.columns = ['variable_1', 'variable_2', 'correlación']
pairwise_correlations = pairwise_correlations.query('variable_1 < variable_2').copy()
pairwise_correlations['correlación_absoluta'] = pairwise_correlations['correlación'].abs()
display(pairwise_correlations.sort_values('correlación_absoluta', ascending=False).head(20))


top_numeric = correlations.head(10).index.tolist()
plt.figure(figsize=(10, 8))
sns.heatmap(numeric_analysis[[TARGET, *top_numeric]].corr(), cmap='vlag', center=0, annot=True, fmt='.2f')
plt.title('Correlaciones de variables numéricas relevantes')
plt.show()

In [ ]:
# Limita Neighborhood a las diez zonas más frecuentes para conservar la legibilidad.
top_neighborhoods = df['Neighborhood'].value_counts().head(10).index
plot_data = df.loc[df['Neighborhood'].isin(top_neighborhoods), ['Neighborhood', TARGET]]

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
sns.boxplot(data=df, x='OverallQual', y=TARGET, ax=axes[0], color='#8cbce4')
axes[0].set(title='SalePrice por calidad general', xlabel='OverallQual', ylabel='Precio de venta (USD)')
sns.boxplot(data=plot_data, x='Neighborhood', y=TARGET, ax=axes[1], color='#8cbce4')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set(title='SalePrice por zona (10 más frecuentes)', xlabel='Neighborhood', ylabel='Precio de venta (USD)')
sns.boxplot(data=df, x='HouseStyle', y=TARGET, ax=axes[2], color='#8cbce4')
axes[2].tick_params(axis='x', rotation=45)
axes[2].set(title='SalePrice por estilo de vivienda', xlabel='HouseStyle', ylabel='Precio de venta (USD)')
plt.tight_layout()
plt.show()

## Frecuencias e interpretación

Las tablas y gráficas de esta sección permiten examinar la concentración de precios, las asimetrías, las variables con mayor correlación y las diferencias visibles entre categorías.

In [ ]:
# Muestra distribuciones de dos variables categóricas relevantes.
for column in ['Neighborhood', 'HouseStyle']:
    frequency = (df[column].value_counts(dropna=False)
                 .rename_axis(column)
                 .to_frame('frecuencia'))
    frequency['porcentaje'] = frequency['frecuencia'] / len(df) * 100
    print(f'Frecuencias: {column}')
    display(frequency.style.format({'porcentaje': '{:.2f}%'}))